# HW5: Question Answering

In [1]:
! pip install -q "transformers>=4.40,<5.0" "tokenizers>=0.19" "sentence-transformers>=2.7" accelerate bert-score datasets rank_bm25 torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.7 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.8 MB/s eta 0:00:00


## 1. Импорты и конфиг

In [3]:
import json
import re
import string
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForQuestionAnswering,
    AutoModelForCausalLM, pipeline
)
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
import torch
from bert_score import BERTScorer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

N_QUERIES = 1000
TOP_K_RERANK = 100
TOP_1 = 1
TOP_5 = 5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

2026-04-23 17:15:51.678381: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776964551.904861      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776964551.967328      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776964552.516339      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776964552.516375      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776964552.516378      55 computation_placer.cc:177] computation placer alr

Device: cuda


## 2. Датасет MIRAGE 

In [4]:
mirage_ds = load_dataset("nlpai-lab/MIRAGE", split="train")
print(f"Full MIRAGE size: {len(mirage_ds)}")
mirage_ds

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/35.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7560 [00:00<?, ? examples/s]

Full MIRAGE size: 7560


Dataset({
    features: ['source', 'query_id', 'query', 'doc_name', 'answer', 'doc_url', 'num_doc_labels', 'doc_pool', 'oracle'],
    num_rows: 7560
})

In [5]:
sample = mirage_ds[0]
print(json.dumps({k: str(v)[:300] for k, v in sample.items()}, ensure_ascii=False, indent=2))

{
  "source": "popqa",
  "query_id": "ce40d2c4-f403-4736-ace1-7fca9c722aba",
  "query": "What is John Mayne's occupation?",
  "doc_name": "John Mayne",
  "answer": "['journalist', 'journo', 'journalists']",
  "doc_url": "https://en.wikipedia.org/wiki?curid=1098597",
  "num_doc_labels": "1",
  "doc_pool": "{'mapped_id': ['ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba'], 'doc_name': ['John Mayne', 'John Mayne', 'John Dawson Mayne', 'John Dawson Mayne', '",
  "oracle": "{'mapped_id': 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'doc_name': 'John Mayne', 'doc_chunk': 'Scottish printer, journalist and poet\\nJohn Mayne (1759–1836) was a Scottish printer, journalist and poet born in Dumfries. In 1780, his poem \"The Siller Gun\" appeared in its original form in \"Ruddiman\\'s M"
}


In [6]:
def get_answers(row):
    ans = row["answer"]
    if isinstance(ans, list):
        return [str(a) for a in ans if a]
    return [str(ans)] if ans else []

def get_oracle_passages(row):
    return row["oracle"].get("doc_chunk", "")

def get_pool_passages(row):
    chunks = row["doc_pool"].get("doc_chunk", [])
    return [c for c in chunks if c]

options_list = [None] * N_QUERIES
indices = list(range(len(mirage_ds)))
random.shuffle(indices)
sub_indices = indices[:N_QUERIES]
mirage_sub = mirage_ds.select(sub_indices)

questions = [row["query"] for row in mirage_sub]
gold_answers = [get_answers(row) for row in mirage_sub]
oracle_passages = [[get_oracle_passages(row)] for row in mirage_sub]
pool_passages = [get_pool_passages(row) for row in mirage_sub]

print(f"Subsample: {len(questions)} вопросов")
print(f"Пример oracle: {oracle_passages[0][0][:200]}")
print(f"Пример pool (кол-во): {len(pool_passages[0])}")

Subsample: 1000 вопросов
Пример oracle: On 6 March 2017, the Kibar nuclear site was captured by the Syrian Democratic Forces – a U.S.-backed coalition of Kurdish and Arab militia fighters – from a retreating ISIL force in northern Deir Ezzo
Пример pool (кол-во): 5


## 3. Метрики

In [ ]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)
    def white_space_fix(text):
        return ' '.join(text.split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return ''.join(ch for ch in text if ch not in exclude)
    return white_space_fix(remove_articles(remove_punc(s.lower())))

def squad_em(prediction, gold_answers):
    pred_norm = normalize_answer(prediction)
    return float(any(normalize_answer(g) == pred_norm for g in gold_answers))

def squad_f1(prediction, gold_answers):
    def token_f1(pred, gold):
        pred_toks = normalize_answer(pred).split()
        gold_toks = normalize_answer(gold).split()
        common = Counter(pred_toks) & Counter(gold_toks)
        num_same = sum(common.values())
        if num_same == 0:
            return 0.0
        precision = num_same / len(pred_toks)
        recall    = num_same / len(gold_toks)
        return 2 * precision * recall / (precision + recall)
    return max(token_f1(prediction, g) for g in gold_answers)

def mirage_em_loose(prediction, gold_answers):
    pred_lower = prediction.lower()
    return float(any(normalize_answer(g) in pred_lower for g in gold_answers))

def mirage_em_strict(prediction, gold_answers):
    return squad_em(prediction, gold_answers)

def compute_squad_metrics(predictions, gold_list):
    em = np.mean([squad_em(p, g) for p, g in zip(predictions, gold_list)])
    f1 = np.mean([squad_f1(p, g) for p, g in zip(predictions, gold_list)])
    return {"EM": round(em, 4), "F1": round(f1, 4)}

def compute_mirage_metrics(predictions, gold_list):
    em_loose  = np.mean([mirage_em_loose(p, g) for p, g in zip(predictions, gold_list)])
    em_strict = np.mean([mirage_em_strict(p, g) for p, g in zip(predictions, gold_list)])
    return {"EM-loose": round(em_loose, 4), "EM-strict": round(em_strict, 4)}

def compute_bert_score(predictions, gold_list, lang="en", batch_size=32):
    references = [g[0] if g else "" for g in gold_list]
    scorer = BERTScorer(
        model_type="bert-base-uncased",
        lang=lang,
        rescale_with_baseline=False,
        use_fast_tokenizer=True,
        batch_size=batch_size,
    )
    P, R, F = scorer.score(predictions, references, verbose=False)
    f1 = round(F.mean().item(), 4)
    return {"BERTScore-F1": f1}

def all_metrics(predictions, gold_list, tag=""):
    sq  = compute_squad_metrics(predictions, gold_list)
    mir = compute_mirage_metrics(predictions, gold_list)
    bs  = compute_bert_score(predictions, gold_list)
    result = {**sq, **mir, **bs}
    print(f"\n{'='*50}")
    print(f" {tag}")
    print(f"{'='*50}")
    for k, v in result.items():
        print(f"  {k:20s}: {v}")
    return result

print("Metrics defined.")

Metrics defined.


## 4. Ранкеры из HW4

Для заданий 4 и 5.  
- **Ranker A**: `ms-marco-MiniLM-L-6-v2`
- **Ranker B**: `multi-qa-MiniLM-L6-cos-v1`

In [ ]:
corpus_dict = {} 
for row in tqdm(mirage_ds, desc="Building corpus"):
    for chunk in get_pool_passages(row):
        pid = str(hash(chunk))
        corpus_dict[pid] = chunk

corpus_ids   = list(corpus_dict.keys())
corpus_texts = [corpus_dict[pid] for pid in corpus_ids]
print(f"Corpus size: {len(corpus_texts)} passages")

Building corpus:   0%|          | 0/7560 [00:00<?, ?it/s]

Corpus size: 36864 passages


In [ ]:
tokenized_corpus = [t.lower().split() for t in tqdm(corpus_texts, desc="Tokenizing for BM25")]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built.")

Tokenizing for BM25:   0%|          | 0/36864 [00:00<?, ?it/s]

BM25 index built.


In [ ]:
from transformers import AutoModelForSequenceClassification

ce_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2")
ce_model     = AutoModelForSequenceClassification.from_pretrained(
    "cross-encoder/ms-marco-MiniLM-L-6-v2").to(DEVICE)
ce_model.eval()

def cross_encoder_score(query, passages, batch_size=32):
    scores = []
    for i in range(0, len(passages), batch_size):
        batch = passages[i:i+batch_size]
        enc = ce_tokenizer(
            [query]*len(batch), batch,
            padding=True, truncation=True, max_length=512,
            return_tensors="pt"
        ).to(DEVICE)
        with torch.no_grad():
            logits = ce_model(**enc).logits.squeeze(-1)
        scores.extend(logits.cpu().tolist())
    return scores

print("Cross-encoder loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Cross-encoder loaded.


In [ ]:
bi_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device=DEVICE)

print("Encoding corpus with bi-encoder...")
corpus_embeddings = bi_model.encode(
    corpus_texts, batch_size=256, show_progress_bar=True,
    convert_to_tensor=True, device=DEVICE
)
print(f"Corpus embeddings shape: {corpus_embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding corpus with bi-encoder...


Batches:   0%|          | 0/144 [00:00<?, ?it/s]

Corpus embeddings shape: torch.Size([36864, 384])


In [ ]:
def retrieve_ranker_a(query, top_k):
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:TOP_K_RERANK]
    candidates = [corpus_texts[i] for i in bm25_top_idx]
    ce_scores  = cross_encoder_score(query, candidates)
    ranked     = sorted(zip(ce_scores, candidates), reverse=True)
    return [t for _, t in ranked[:top_k]]

def retrieve_ranker_b(query, top_k):
    q_emb  = bi_model.encode(query, convert_to_tensor=True, device=DEVICE)
    scores = util.cos_sim(q_emb, corpus_embeddings)[0]
    top_idx = scores.argsort(descending=True)[:top_k].cpu().tolist()
    return [corpus_texts[i] for i in top_idx]


test_q = questions[0]
print("Test Q:", test_q)
print("Ranker A top-1:", retrieve_ranker_a(test_q, 1)[0][:200])
print("Ranker B top-1:", retrieve_ranker_b(test_q, 1)[0][:200])

Test Q: If the Mossad, the Israeli intelligence agency, had proceeded with their initial plan to assassinate a high-ranking Syrian official in London, who would have been the targeted individual?
Ranker A top-1: On 6 March 2017, the Kibar nuclear site was captured by the Syrian Democratic Forces – a U.S.-backed coalition of Kurdish and Arab militia fighters – from a retreating ISIL force in northern Deir Ezzo
Ranker B top-1: On 6 March 2017, the Kibar nuclear site was captured by the Syrian Democratic Forces – a U.S.-backed coalition of Kurdish and Arab militia fighters – from a retreating ISIL force in northern Deir Ezzo


## 5. Task 1: Base LM, 0-shot (no context)

In [ ]:
SLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

slm_tokenizer = AutoTokenizer.from_pretrained(SLM_MODEL)
slm_model     = AutoModelForCausalLM.from_pretrained(
    SLM_MODEL, torch_dtype=torch.float16 if DEVICE=="cuda" else torch.float32,
    device_map="auto"
)
slm_pipe = pipeline(
    "text-generation", model=slm_model, tokenizer=slm_tokenizer,
    max_new_tokens=64, max_length=None, do_sample=False, temperature=None, top_p=None
)
print(f"{SLM_MODEL} loaded.")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen/Qwen2.5-1.5B-Instruct loaded.


In [ ]:
def build_prompt_0shot(question):
    return (
        "Answer the following question with a short phrase or a single word. "
        "Give only the answer, no explanation.\n\n"
        f"Question: {question}\nAnswer:"
    )

def extract_answer_from_generation(generated, prompt):
    answer = generated[len(prompt):].strip()
    return answer.split("\n")[0].strip()

def generate_answers(prompts, desc="Generating"):
    answers = []
    for prompt in tqdm(prompts, desc=desc):
        out = slm_pipe(prompt)[0]["generated_text"]
        answers.append(extract_answer_from_generation(out, prompt))
    return answers

In [ ]:
prompts_0shot = [build_prompt_0shot(q) for q in questions]
preds_task1   = generate_answers(prompts_0shot, desc="Task 1 (0-shot)")

Task 1 (0-shot):   0%|          | 0/1000 [00:00<?, ?it/s]

In [19]:
metrics_task1 = all_metrics(preds_task1, gold_answers, tag="Task 1: Base LM 0-shot")


 Task 1: Base LM 0-shot
  EM                  : 0.022
  F1                  : 0.0669
  EM-loose            : 0.031
  EM-strict           : 0.022
  BERTScore-F1        : 0.4802


## 6. Task 2: Extractive QA

In [8]:
from transformers import AutoModelForQuestionAnswering

EXTRACTIVE_MODEL = "deepset/roberta-base-squad2"

qa_tokenizer = AutoTokenizer.from_pretrained(EXTRACTIVE_MODEL, use_fast=False)
qa_model = AutoModelForQuestionAnswering.from_pretrained(EXTRACTIVE_MODEL)

qa_pipe = pipeline(
    "question-answering",
    model=qa_model,
    tokenizer=qa_tokenizer,
    device=0 if DEVICE=="cuda" else -1
)
print(f"{EXTRACTIVE_MODEL} loaded.")

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Device set to use cuda:0


deepset/roberta-base-squad2 loaded.


In [ ]:
def extractive_qa_oracle(question, passages):
    if not passages:
        return ""
    best_score = -1e9
    best_answer = ""
    for passage in passages:
        if not passage.strip():
            continue
        try:
            result = qa_pipe(
                question=question, context=passage,
                max_answer_len=50,
                handle_impossible_answer=False,
            )
            if result["score"] > best_score:
                best_score  = result["score"]
                best_answer = result["answer"]
        except Exception as e:
            print(f"QA error: {e}")
    return best_answer

preds_task2 = [
    extractive_qa_oracle(q, p)
    for q, p in tqdm(zip(questions, oracle_passages), total=len(questions), desc="Task 2 (extractive)")
]

Task 2 (extractive):   0%|          | 0/1000 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [18]:
metrics_task2 = all_metrics(preds_task2, gold_answers, tag="Task 2: Extractive QA (oracle)")


 Task 2: Extractive QA (oracle)
  EM                  : 0.615
  F1                  : 0.7324
  EM-loose            : 0.658
  EM-strict           : 0.615
  BERTScore-F1        : 0.7418


## 7. Task 3: Open-book QA

In [ ]:
def build_prompt_openbook(question, passages, max_chars=2000):
    context = "\n\n".join(passages)[:max_chars]
    return (
        "Use the following passages to answer the question concisely. "
        "Give only the answer, no explanation.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

prompts_task3 = [
    build_prompt_openbook(q, p)
    for q, p in zip(questions, oracle_passages)
]
preds_task3 = generate_answers(prompts_task3, desc="Task 3 (open-book, oracle)")

Task 3 (open-book, oracle):   0%|          | 0/1000 [00:00<?, ?it/s]

In [17]:
metrics_task3 = all_metrics(preds_task3, gold_answers, tag="Task 3: Open-book QA (oracle)")


 Task 3: Open-book QA (oracle)
  EM                  : 0.644
  F1                  : 0.773
  EM-loose            : 0.738
  EM-strict           : 0.644
  BERTScore-F1        : 0.8159


## 8. Task 4: Top-1 passage (rankers from HW4)

In [ ]:
top1_a, top1_b = [], []

for q in tqdm(questions, desc="Retrieve top-1 (Ranker A: cross-encoder)"):
    top1_a.append(retrieve_ranker_a(q, TOP_1))

for q in tqdm(questions, desc="Retrieve top-1 (Ranker B: bi-encoder)"):
    top1_b.append(retrieve_ranker_b(q, TOP_1))

Retrieve top-1 (Ranker A: cross-encoder):   0%|          | 0/1000 [00:00<?, ?it/s]

Retrieve top-1 (Ranker B: bi-encoder):   0%|          | 0/1000 [00:00<?, ?it/s]

In [27]:
def build_prompt_with_context(question: str, passages: list, max_chars: int = 2000) -> str:
    context = "\n\n".join(passages)[:max_chars]
    return (
        "Use the following passage to answer the question concisely. "
        "Give only the answer, no explanation.\n\n"
        f"Passage:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

In [ ]:
prompts_t4_a = [build_prompt_with_context(q, p) for q, p in zip(questions, top1_a)]
preds_t4_a   = generate_answers(prompts_t4_a, desc="Task 4 – Ranker A top-1")
metrics_t4_a = all_metrics(preds_t4_a, gold_answers,
                            tag="Task 4: Ranker A (cross-encoder) top-1")

Task 4 – Ranker A top-1:   0%|          | 0/1000 [00:00<?, ?it/s]


 Task 4: Ranker A (cross-encoder) top-1
  EM                  : 0.344
  F1                  : 0.4405
  EM-loose            : 0.39
  EM-strict           : 0.344
  BERTScore-F1        : 0.6445


In [ ]:
prompts_t4_b = [build_prompt_with_context(q, p) for q, p in zip(questions, top1_b)]
preds_t4_b   = generate_answers(prompts_t4_b, desc="Task 4 – Ranker B top-1")
metrics_t4_b = all_metrics(preds_t4_b, gold_answers,
                            tag="Task 4: Ranker B (bi-encoder) top-1")

Task 4 – Ranker B top-1:   0%|          | 0/1000 [00:00<?, ?it/s]


 Task 4: Ranker B (bi-encoder) top-1
  EM                  : 0.365
  F1                  : 0.4588
  EM-loose            : 0.403
  EM-strict           : 0.365
  BERTScore-F1        : 0.6502


## 9. Task 5: Top-5 passages (ranker context vs MIRAGE mixed context)

In [ ]:
top5_a, top5_b = [], []

for q in tqdm(questions, desc="Retrieve top-5 (Ranker A)"):
    top5_a.append(retrieve_ranker_a(q, TOP_5))

for q in tqdm(questions, desc="Retrieve top-5 (Ranker B)"):
    top5_b.append(retrieve_ranker_b(q, TOP_5))

Retrieve top-5 (Ranker A):   0%|          | 0/1000 [00:00<?, ?it/s]

Retrieve top-5 (Ranker B):   0%|          | 0/1000 [00:00<?, ?it/s]

In [32]:
def build_prompt_multi(question: str, passages: list, max_chars: int = 3000) -> str:
    context = "\n\n".join(
        [f"[Passage {i+1}] {p}" for i, p in enumerate(passages)]
    )[:max_chars]
    return (
        "Use the following passages to answer the question concisely. "
        "Give only the answer, no explanation.\n\n"
        f"Passages:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

In [ ]:
prompts_t5_a = [build_prompt_multi(q, p) for q, p in zip(questions, top5_a)]
preds_t5_a   = generate_answers(prompts_t5_a, desc="Task 5 – Ranker A top-5")
metrics_t5_a = all_metrics(preds_t5_a, gold_answers,
                            tag="Task 5: Ranker A (cross-encoder) top-5")

Task 5 – Ranker A top-5:   0%|          | 0/1000 [00:00<?, ?it/s]


 Task 5: Ranker A (cross-encoder) top-5
  EM                  : 0.265
  F1                  : 0.361
  EM-loose            : 0.387
  EM-strict           : 0.265
  BERTScore-F1        : 0.5828


In [ ]:
prompts_t5_b = [build_prompt_multi(q, p) for q, p in zip(questions, top5_b)]
preds_t5_b   = generate_answers(prompts_t5_b, desc="Task 5 – Ranker B top-5")
metrics_t5_b = all_metrics(preds_t5_b, gold_answers,
                            tag="Task 5: Ranker B (bi-encoder) top-5")

Task 5 – Ranker B top-5:   0%|          | 0/1000 [00:00<?, ?it/s]


 Task 5: Ranker B (bi-encoder) top-5
  EM                  : 0.33
  F1                  : 0.4343
  EM-loose            : 0.482
  EM-strict           : 0.33
  BERTScore-F1        : 0.6296


In [ ]:
mirage_top5 = [random.sample(p, min(5, len(p))) for p in pool_passages]

prompts_t5_mirage = [build_prompt_multi(q, p) for q, p in zip(questions, mirage_top5)]
preds_t5_mirage   = generate_answers(prompts_t5_mirage, desc="Task 5 – MIRAGE mixed context")
metrics_t5_mirage = all_metrics(preds_t5_mirage, gold_answers,
                                 tag="Task 5: MIRAGE mixed context (oracle top-5)")

Task 5 – MIRAGE mixed context:   0%|          | 0/1000 [00:00<?, ?it/s]


 Task 5: MIRAGE mixed context (oracle top-5)
  EM                  : 0.31
  F1                  : 0.4059
  EM-loose            : 0.44
  EM-strict           : 0.31
  BERTScore-F1        : 0.6155


## 10. Summary table

In [49]:
results = {
    "T1: Base LM 0-shot":                metrics_task1,
    "T2: Extractive QA (oracle)": metrics_task2,
    "T3: Open-book (oracle)": metrics_task3,
    "T4: Ranker A top-1": metrics_t4_a,
    "T4: Ranker B top-1": metrics_t4_b,
    "T5: Ranker A top-5": metrics_t5_a,
    "T5: Ranker B top-5": metrics_t5_b,
    "T5: MIRAGE mixed top-5":  metrics_t5_mirage,
}

df = pd.DataFrame(results).T
df.index.name = "Configuration"
print(df.to_string())

                               EM      F1  EM-loose  EM-strict  BERTScore-F1
Configuration                                                               
T1: Base LM 0-shot          0.022  0.0669     0.031      0.022        0.4802
T2: Extractive QA (oracle)  0.615  0.7324     0.658      0.615        0.7418
T3: Open-book (oracle)      0.644  0.7730     0.738      0.644        0.8159
T4: Ranker A top-1          0.344  0.4405     0.390      0.344        0.6445
T4: Ranker B top-1          0.365  0.4588     0.403      0.365        0.6502
T5: Ranker A top-5          0.265  0.3610     0.387      0.265        0.5828
T5: Ranker B top-5          0.330  0.4343     0.482      0.330        0.6296
T5: MIRAGE mixed top-5      0.310  0.4059     0.440      0.310        0.6155


In [ ]:
df.to_csv("hw5_results.csv")
print("Results saved to hw5_results.csv")

Results saved to hw5_results.csv
